# Migrasjonsmønstre — Folkeregister

Analyserer interne migrasjonsmønstre: hvilke kommuner tiltrekker innflyttere, hvilke mister befolkning, og de største flytte-strømmene.

**Kilde:** `folkeregister.migration_flows`, `folkeregister.population_stats` (Gold-lag)  
**Relatert issue:** US E-2 / #94

In [ ]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL delta;  LOAD delta;")

_raw = os.environ.get("MINIO_ENDPOINT", "http://minio.slettix-analytics.svc.cluster.local:9000")
_endpoint = _raw.replace("http://", "").replace("https://", "")

con.execute(f"""
    CREATE OR REPLACE SECRET minio (
        TYPE      S3,
        KEY_ID    '{os.environ.get("MINIO_ACCESS_KEY", "admin")}',
        SECRET    '{os.environ.get("MINIO_SECRET_KEY", "changeme")}',
        ENDPOINT  '{_endpoint}',
        URL_STYLE 'path',
        USE_SSL   false
    );
""")

MIGRATION_PATH   = "s3://gold/folkeregister/migration_flows"
POPULATION_PATH  = "s3://gold/folkeregister/population_stats"
print("DuckDB klar.", duckdb.__version__)

In [ ]:
df_mig = con.execute(f"SELECT * FROM delta_scan('{MIGRATION_PATH}')").df()
df_pop = con.execute(f"SELECT * FROM delta_scan('{POPULATION_PATH}')").df()
print(f"Migrasjonsrader: {len(df_mig):,} | Unike fra-kommuner: {df_mig['from_municipality_code'].nunique()}")
df_mig.head()

## 1. Topp 10 migrasjonsstrømmer (totalt over alle år)

In [ ]:
top_flows = (
    df_mig
    .groupby(["from_municipality_name", "to_municipality_name"])["flow_count"]
    .sum()
    .reset_index()
    .sort_values("flow_count", ascending=False)
    .head(10)
)
top_flows["strøm"] = top_flows["from_municipality_name"] + " → " + top_flows["to_municipality_name"]
top_flows = top_flows.sort_values("flow_count")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_flows["strøm"], top_flows["flow_count"], color="steelblue")
for bar, val in zip(bars, top_flows["flow_count"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=8)
ax.set_xlabel("Antall flyttinger (totalt)")
ax.set_title("Topp 10 migrasjonsstrømmer", fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Netto migrasjon per kommune

In [ ]:
# Netto migrasjon = innflytting - utflytting per kommune
inn  = df_mig.groupby("to_municipality_name")["flow_count"].sum().rename("inn")
ut   = df_mig.groupby("from_municipality_name")["flow_count"].sum().rename("ut")
netto = (inn - ut).fillna(0).sort_values()

# Vis kommuner med størst absolutt netto-migrasjon (topp/bunn 15)
vis = pd.concat([netto.nsmallest(8), netto.nlargest(8)]).drop_duplicates().sort_values()
colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in vis]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(vis.index, vis.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Netto migrasjon (innflytting − utflytting)")
ax.set_title("Netto migrasjon per kommune — topp og bunn 8", fontweight="bold")
inn_patch = mpatches.Patch(color="#2ecc71", label="Netto innflytting")
ut_patch  = mpatches.Patch(color="#e74c3c", label="Netto utflytting")
ax.legend(handles=[inn_patch, ut_patch])
plt.tight_layout()
plt.show()

## 3. Netto migrasjonsutvikling over tid — topp 5 vekst-kommuner

In [ ]:
# Kommuner med høyest total netto innflytting
top5 = netto.nlargest(5).index.tolist()

inn_yr = df_mig.groupby(["to_municipality_name",   "reference_year"])["flow_count"].sum().rename("inn")
ut_yr  = df_mig.groupby(["from_municipality_name", "reference_year"])["flow_count"].sum().rename("ut")
netto_yr = (inn_yr - ut_yr).fillna(0).reset_index()
netto_yr.columns = ["municipality", "year", "net"]

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.get_cmap("tab10")
for i, kom in enumerate(top5):
    sub = netto_yr[netto_yr["municipality"] == kom].sort_values("year")
    ax.plot(sub["year"], sub["net"], marker="o", color=cmap(i), label=kom)

ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_xlabel("År")
ax.set_ylabel("Netto migrasjon")
ax.set_title("Netto migrasjonsutvikling — topp 5 vekstkommuner", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Sankey-diagram — de 10 største strømmene

Krever `plotly`. Installeres automatisk hvis ikke tilgjengelig.

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "-q"])
    import plotly.graph_objects as go

flows = (
    df_mig
    .groupby(["from_municipality_name", "to_municipality_name"])["flow_count"]
    .sum()
    .reset_index()
    .sort_values("flow_count", ascending=False)
    .head(12)
)

all_nodes = pd.unique(flows[["from_municipality_name", "to_municipality_name"]].values.ravel())
node_idx  = {n: i for i, n in enumerate(all_nodes)}

fig = go.Figure(go.Sankey(
    node=dict(label=list(all_nodes), pad=15, thickness=20,
              color=["steelblue"] * len(all_nodes)),
    link=dict(
        source=[node_idx[r] for r in flows["from_municipality_name"]],
        target=[node_idx[r] for r in flows["to_municipality_name"]],
        value=flows["flow_count"].tolist(),
    )
))
fig.update_layout(
    title_text="Topp 12 migrasjonsstrømmer (Sankey)",
    font_size=12,
    height=500,
)
fig.show()